### Playing with data
Aims of exploratory data anlysis, explore relationships between:
- Rainfall descriptors (event volume, is this the only rainfall characteristic?)
- Flooding outcomes ()
- Catchment descriptors
- Antecedent soil moisture conditions (mean value over catchment?, would there be a way to find mean value more specifically over the peak of the rainfall event )

In [2]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
from scipy import stats
import os
import numpy as np
import iris
import contextily as ctx

home_dir = "C:/Users/kv25483/OneDrive - University of Bristol/FutureFlood/Data/"
home_dir = "/scratch/hydro4/users/" # kv25483/"

## Get catchment data

In [2]:
catchments = pd.read_csv(home_dir + "kv25483/FutureFlood/Data/NewcastleExample/catchment_identifier/hyd_areas_GB_with_subcatchments_no_spaces.csv")

In [3]:
catchments = gpd.read_file(home_dir + "kv25483/FutureFlood/Data/NewcastleExample/catchment_identifier/hyd_areas_GB_with_subcatchments.shp")
newcastle = catchments[catchments['HA_NUM'] == "23"]

## Get rainfall data
This contains 95 rows, relating to 95 events which had rainfall which was over the filtering threshold

In [3]:
ncl_events = pd.read_csv(home_dir + "la17355/FUTURE-FLOOD/UKCP_rainfall_events/fixed_threshold_30mm_with_volume/Tyne(Northumberland)_01_full_events_with_event_nums.csv")
ncl_events.head()

,start_indices,stop_indices,event_durations,peaks,start_year,start_month,start_day,start_hour,start_time,stop_time,start_time_seconds,stop_time_seconds,event_vol,max_extent,acc_total,acc_3hr,acc_6hr,acc_12hr,acc_24hr,event_num
0,6177,6194,17,30.50427,1992,8,18,8,195536.5,195553.5,703931400,703992600,1853.791106,73,1853.791106,1070.589100,1552.028500,1847.382087,NaN,1
1,5293,5303,10,32.84921,1994,7,11,12,211932.5,211942.5,762957000,762993000,126.233498,5,126.233498,115.582377,126.232947,NaN,NaN,2
2,5384,5411,27,56.25648,1994,7,15,7,212023.5,212050.5,763284600,763381800,1854.094660,75,1854.094660,586.499210,857.442260,1223.954138,1847.511375,3
3,6829,6850,21,35.77378,1995,9,15,12,222108.5,222129.5,799590600,799666200,3123.716481,88,3123.716481,1857.628600,2773.342800,3080.387700,NaN,4
4,4236,4250,14,39.58595,2001,5,27,11,271355.5,271369.5,976879800,976930200,461.172221,17,461.172221,228.737340,386.248000,461.171202,NaN,5


## Get soil data
23 is catchment number of Newcastle
soil_{HA_NUM}_{event_num}_{year}_{month}_{day}_Ens_{ensemble_member}_vol_30m.tif 

Order files into event number order, and then loop through each file finding the maean, max and min soil moisture over the whole catchment.

In [7]:
files = os.listdir(home_dir + "kv25483/FutureFlood/Data/NewcastleExample/soil/Ens01_23")
def get_event_num(filename):
    parts = filename.replace(".tif", "").split("_")
    return int(parts[2])   # <-- event_num position
sorted_files = sorted(files, key=get_event_num)

mean_soil_moistures = []
max_soil_moistures = []
min_soil_moistures = []
p80_soil_moistures = []
p90_soil_moistures = []

for file in sorted_files:
    with rasterio.open(home_dir + f"kv25483/FutureFlood/Data/NewcastleExample/soil/Ens01_23/{file}") as src:
        # print(src.width, src.height)
        # print(src.count)  # number of bands
        img = src.read(1)  # read first band
        # Replace -9999 (or src.nodata) with np.nan
        img = img.astype(float)  # ensure it's float, so np.nan works
        img[img == -9999] = np.nan
        
        # Now compute the mean, ignoring NaNs
        mean_soil_moisture = round(np.nanmean(img),2)
        max_soil_moisture = round(np.nanmax(img),2)
        min_soil_moisture = round(np.nanmin(img),2)
        p80_soil_moisture  = round(np.nanpercentile(img, 80), 2)
        p90_soil_moisture  = round(np.nanpercentile(img, 90), 2)

        # Append to list
        mean_soil_moistures.append(mean_soil_moisture)
        p80_soil_moistures.append(p80_soil_moisture)
        p90_soil_moistures.append(p90_soil_moisture)
        max_soil_moistures.append(max_soil_moisture)
        min_soil_moistures.append(min_soil_moisture)

# add to event dataframe
ncl_events['mean_soil_moisture'] = mean_soil_moistures
ncl_events['max_soil_moisture'] = max_soil_moistures
ncl_events['mean_soil_moisture'] = mean_soil_moistures
ncl_events['p80_soil_moisture'] = p80_soil_moistures
ncl_events['p90_soil_moisture'] = p90_soil_moistures

### Get flood data

In [8]:
# test =pd.read_csv(home_dir + "kv25483/FutureFlood/Data/NewcastleExample/flooded_area/Ens01_23/Tyne(Northumberland)_01_full_events_with_event_nums_flooded_area.csv")
# test['date'] = pd.to_datetime(dict(year=test['start_year'], month=test['start_month'],day=test['start_day']))
# test['max_extent']
# # ncl_events['flood_area_km2_10cm'] = test['flood_area_km2_10cm']
# # ncl_events['flood_area_perc_10cm'] = test['flood_area_perc_10cm']

In [5]:
# soil_tester_fp = home_dir + f"kv25483/FutureFlood/Data/NewcastleExample/soil/Ens01_23/{file}"
# src = rasterio.open(soil_tester_fp)

# # Check CRS
# print("Raster CRS:", src.crs, "Catchment CRS:", newcastle.crs)

# # Reproject catchment if needed
# if newcastle.crs != src.crs:
#     newcastle = newcastle.to_crs(src.crs)

# fig, ax = plt.subplots(figsize=(4, 4))
# # Plot catchment boundary
# newcastle.boundary.plot(ax=ax, edgecolor="red", linewidth=2)
# ax.set_title("Soil Moisture Raster with Newcastle Catchment")
# ctx.add_basemap(ax, crs=src.crs)
# # Plot raster
# show(src, ax=ax, cmap="viridis")

## Test relationships between variables

In [9]:
# x = ncl_events['acc_3hr']
# y = ncl_events['flood_area_perc_10cm']

# slope, intercept, r, p, std_err = stats.linregress(x, y)

# def myfunc(x):
#     return slope * x + intercept

# mymodel = list(map(myfunc, x))

# plt.scatter(x, y, color='black')
# plt.plot(x, mymodel)
# plt.show()